##### Boilerplate

In [ ]:
# @title
from google.colab import drive
drive.mount('/content/drive')

import os
import sys

PROJECT_DIR = "/content/drive/MyDrive/DumLLM"

os.chdir(PROJECT_DIR)

sys.path.insert(0, os.path.join(PROJECT_DIR, "src"))

print("DumLLM initialized!")
print("Working directory:", os.getcwd())
print("Project files:", os.listdir(PROJECT_DIR))

Mounted at /content/drive
DumLLM initialized!
Working directory: /content/drive/MyDrive/DumLLM
Project files: ['requirements.txt', '.venv', '.git', 'notebooks', 'public', '.gitignore', 'start_DumLLM.ipynb']


# 4. Implementing a GPT Model from scratch to generate text

## 4.1 Coding an LLM Architecture

In [2]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,        # Vocabulary size
    "context_length": 1024,     # Context length
    "emb_dim" : 768,            # Embedding dimension
    "n_head": 12,               # Number of attention heads
    "n_layer": 12,              # Number of layers (TransformerBlocks)
    "drop_rate" : 0.1,          # Dropout rate
    "qkv_bais" : False          # Query-Key-Value bias
}

In [3]:
import torch
import torch.nn as nn

class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
      super().__init__()
      self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
      self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
      self.drop_emb = nn.Dropout(cfg["drop_rate"])

      # use a placeholder for TransformerBlock
      self.trf_blocks = nn.Sequential(
          *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layer"])]
      )

      # use a placeholder for LayerNorm
      self.final_norm = DummyLayerNorm(cfg["emb_dim"])
      self.out_head = nn.Linear(
          cfg["emb_dim"], cfg["vocab_size"], bias=False
      )

    def forward(self, in_idx):
      batch_size, seq_len = in_idx.shape
      tok_embeds = self.tok_emb(in_idx)
      pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
      x = tok_embeds + pos_embeds
      x = self.drop_emb(x)
      x = self.trf_blocks(x)
      x = self.final_norm(x)
      logits = self.out_head(x)
      return logits

class DummyTransformerBlock(nn.Module):
  def __init__(self, cfg):
    super().__init__()

  def forward(self, x):
    return x

class DummyLayerNorm(nn.Module):
  def __init__(self, normalized_shape, eps=1e-5):
    super().__init__()

  def forward(self, x):
    return x

In [5]:
import tiktoken

tokenizer = tiktoken.encoding_for_model("gpt2")

batch = []

txt1 = "Every effort moves you"
txt2 = "Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))

batch = torch.stack(batch, dim=0)
print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


In [6]:
torch.manual_seed(123)
model = DummyGPTModel(cfg=GPT_CONFIG_124M)

logits = model(batch)
print("Output shape: ", logits.shape)
print(logits)

Output shape:  torch.Size([2, 4, 50257])
tensor([[[-1.2034,  0.3201, -0.7130,  ..., -1.5548, -0.2390, -0.4667],
         [-0.1192,  0.4539, -0.4432,  ...,  0.2392,  1.3469,  1.2430],
         [ 0.5307,  1.6720, -0.4695,  ...,  1.1966,  0.0111,  0.5835],
         [ 0.0139,  1.6755, -0.3388,  ...,  1.1586, -0.0435, -1.0400]],

        [[-1.0908,  0.1798, -0.9484,  ..., -1.6047,  0.2439, -0.4530],
         [-0.7860,  0.5581, -0.0610,  ...,  0.4835, -0.0077,  1.6621],
         [ 0.3567,  1.2698, -0.6398,  ..., -0.0162, -0.1296,  0.3717],
         [-0.2407, -0.7349, -0.5102,  ...,  2.0057, -0.3694,  0.1814]]],
       grad_fn=<UnsafeViewBackward0>)
